# Gipformer 1.5 68M — Colab T4

**Model:** [g-group-ai-lab/gipformer1.5-68M-rnnt](https://huggingface.co/g-group-ai-lab/gipformer1.5-68M-rnnt)

Sử dụng PyTorch CUDA theo repository chính thức, đúng phiên bản **1.5**. Kết quả Time/RTF của CLI và tổng wall time được tách riêng.

## 1. Kiểm tra Colab T4
Runtime → Change runtime type → T4 GPU. Cell dưới sẽ dừng nếu không có GPU.

In [1]:
import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Chưa có CUDA: hãy chọn T4 GPU và kết nối lại runtime"
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


## 2. Clone repository và cài CUDA dependencies chính thức

In [2]:
%cd /content
!git clone -q https://github.com/ggroup-ai-lab/gipformer.git || true
%cd /content/gipformer
%pip -q install uv soundfile
!uv sync --extra pytorch

/content
/content/gipformer
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 87.1 MB/s eta 0:00:00
Using CPython 3.13.15 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 98 packages in 5.10s
Prepared 58 packages in 1m 15s
Installed 58 packages in 469ms
 + absl-py==2.5.1
 + anyio==4.15.1
 + cffi==2.1.1
 + click==8.5.0
 + filelock==4.1.1
 + fsspec==2026.9.0
 + graphviz==0.21
 + grpcio==1.84.0
 + h11==0.16.0
 + hf-xet==1.7.0
 + httpcore2==2.13.1
 + httpx2==2.13.1
 + huggingface-hub==2.2.0
 + idna==3.20
 + jinja2==3.1.6
 + k2==1.24.4.dev20250807+cuda12.8.torch2.8.0
 + kaldialign==0.12.0
 + kaldifeat==1.25.5.dev20250807+cuda12.8.torch2.8.0
 + markdown==3.11.1
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + networkx==3.7
 + numpy==2.5.4
 + nvidia-cublas-cu12==12.8.4.1
 + nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufile-cu12==

## 3. Upload và chuẩn hóa file audio

In [3]:
from google.colab import files
from pathlib import Path
import subprocess, soundfile as sf
uploaded = files.upload()
assert uploaded, "Hãy upload ít nhất một file audio"
original = next(iter(uploaded))
audio_path = '/content/asr_test_16k.wav'
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',original,'-ac','1','-ar','16000','-c:a','pcm_s16le',audio_path],check=True)
signal,sr = sf.read(audio_path)
duration = len(signal)/sr
print(f'Audio: {audio_path} | {duration:.2f}s | {sr}Hz')

Saving p_5694761_156.mp3 to p_5694761_156.mp3
Audio: /content/asr_test_16k.wav | 10.35s | 16000Hz


## 4. Kiểm tra CUDA trong `.venv` của Gipformer

In [4]:
!uv run --extra pytorch python -c "import torch; print('Torch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.is_available()); assert torch.cuda.is_available()"

Torch: 2.8.0+cu128 CUDA: 12.8 GPU: True


## 5. Chạy Gipformer **phiên bản 1.5** qua CLI chính thức
Lệnh này tải checkpoint và chạy GPU. `--version 1.5` là cần thiết để không dùng nhầm Gipformer v1.

In [5]:
!uv run --extra pytorch python infer_pytorch.py --audio /content/asr_test_16k.wav --version 1.5 --device cuda --decoding-method modified_beam_search

Setting up icefall (one-time download)...
Cloning into '/root/.cache/gipformer/icefall'...
remote: Enumerating objects: 364, done.
remote: Counting objects: 100% (364/364), done.
remote: Compressing objects: 100% (286/286), done.
remote: Total 364 (delta 41), reused 194 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (364/364), 112.39 KiB | 4.32 MiB/s, done.
Resolving deltas: 100% (41/41), done.
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 12 (delta 0), reused 2 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 11.61 KiB | 1.45 MiB/s, done.
remote: Enumerating objects: 631, done.
remote: Counting objects: 100% (631/631), done.
remote: Compressing objects: 100% (385/385), done.
remote: Total 631 (delta 295), reused 340 (delta 171), pack-reused 0 (from 0)
Receiving objects: 100% (631/631), 962.28 KiB | 7.89 MiB/s, done.
Resolving deltas: 100% (295/295), done.

## 6. Đo tổng wall time (gồm khởi động Python và load model)
**Không nhầm** thời gian bên dưới với `Time` và `RTF` mà script in ra. Mỗi lần subprocess sẽ nạp model lại.

In [6]:
import subprocess, time
cmd=['uv','run','--extra','pytorch','python','infer_pytorch.py',
     '--audio','/content/asr_test_16k.wav','--version','1.5','--device','cuda',
     '--decoding-method','modified_beam_search']
t0=time.perf_counter()
p=subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,check=True)
print(p.stdout)
print(f'Wall time (new process + load + inference): {time.perf_counter()-t0:.3f}s')

Model downloaded successfully.
Using device: cuda
Model parameters: 68,625,511

  File: /content/asr_test_16k.wav
  Text: VĂN BẢN NÀY CÓ THỂ ĐƯỢC THAY THẾ BẰNG BẤT KỲ VĂN BẢN NÀO TRONG HỘP VĂN BẢN NÀY BẠN CÓ THỂ NHẬP HOẶC DÁN VĂN BẢN CỦA RIÊNG MÌNH BẠN CÓ THỂ SỬ DỤNG NHIỀU GIỌNG NÓI CHO MỘT VĂN BẢN
  Time: 1.10s | Audio: 10.35s | RTF: 0.107

Wall time (new process + load + inference): 5.907s


## 7. Lưu kết quả từ CLI
Script chính thức cung cấp `Time/Audio/RTF`; các thông số này đo vùng xử lý do chính script xác định. Để benchmark warm-request tương đương Zipformer, cần giữ model trong cùng Python process và đo cùng các bước (không dùng subprocess mỗi request).

In [7]:
import re, json
from pathlib import Path
lines=p.stdout.splitlines()
info={"model":"g-group-ai-lab/gipformer1.5-68M-rnnt","runtime":"official PyTorch CUDA CLI","audio_file":"/content/asr_test_16k.wav"}
for line in lines:
    s=line.strip()
    if s.startswith('Text:'): info['text']=s.partition(':')[2].strip()
    if s.startswith('Time:'):
        m=re.search(r'Time:\s*([0-9.]+)s\s*\|\s*Audio:\s*([0-9.]+)s\s*\|\s*RTF:\s*([0-9.]+)',s)
        if m: info.update({'script_time_seconds':float(m.group(1)),'audio_seconds':float(m.group(2)),'script_rtf':float(m.group(3))})
print(info)
Path('/content/gipformer15_results.json').write_text(json.dumps(info,ensure_ascii=False,indent=2))

{'model': 'g-group-ai-lab/gipformer1.5-68M-rnnt', 'runtime': 'official PyTorch CUDA CLI', 'audio_file': '/content/asr_test_16k.wav', 'text': 'VĂN BẢN NÀY CÓ THỂ ĐƯỢC THAY THẾ BẰNG BẤT KỲ VĂN BẢN NÀO TRONG HỘP VĂN BẢN NÀY BẠN CÓ THỂ NHẬP HOẶC DÁN VĂN BẢN CỦA RIÊNG MÌNH BẠN CÓ THỂ SỬ DỤNG NHIỀU GIỌNG NÓI CHO MỘT VĂN BẢN', 'script_time_seconds': 1.1, 'audio_seconds': 10.35, 'script_rtf': 0.107}


410